# Exploration Betti0

**Rappel** : `Betti0 = |n_composantes_pred - n_composantes_gt|`

Ce notebook charge une prédiction, calcule le Betti0 par rapport au GT, et visualise chaque composante connexe avec une couleur distincte pour comprendre d'où vient l'écart.

## Chemins — à adapter selon le cluster

In [ ]:
import os

# ── À adapter ──────────────────────────────────────────────────────────────
# Dossier contenant les .nii.gz produits par nnUNetv2_predict
PRED_DIR = "/path/to/predictions"

# GT de test (labelsTs du dataset source)
GT_DIR = "/path/to/nnUNet_raw/Dataset100_PARSE/labelsTs"
# ──────────────────────────────────────────────────────────────────────────

assert os.path.isdir(PRED_DIR), f"PRED_DIR introuvable : {PRED_DIR}"
assert os.path.isdir(GT_DIR),   f"GT_DIR introuvable : {GT_DIR}"

## Vue d'ensemble — Betti0 sur tous les cas

In [ ]:
import glob
import numpy as np
import nibabel as nib
import pandas as pd
from scipy.ndimage import label as label_cc

pred_files = sorted(glob.glob(f"{PRED_DIR}/*.nii.gz"))
print(f"{len(pred_files)} prédictions trouvées")

rows = []
for pred_path in pred_files:
    fname = os.path.basename(pred_path)
    gt_path = os.path.join(GT_DIR, fname)
    if not os.path.exists(gt_path):
        print(f"  [WARN] GT manquant pour {fname}")
        continue

    pred = nib.load(pred_path).get_fdata().astype(bool)
    gt   = nib.load(gt_path).get_fdata().astype(bool)

    _, n_pred = label_cc(pred)
    _, n_gt   = label_cc(gt)

    rows.append({
        "case":   fname,
        "n_gt":   n_gt,
        "n_pred": n_pred,
        "betti0": abs(n_pred - n_gt),
        "delta":  n_pred - n_gt,       # positif = trop de composantes, négatif = manquantes
    })

df = pd.DataFrame(rows).sort_values("betti0", ascending=False)
print(df.to_string(index=False))
print(f"\nBetti0 moyen : {df['betti0'].mean():.2f}  |  max : {df['betti0'].max()}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Distribution du Betti0
axes[0].bar(df["case"].str.replace(".nii.gz", ""), df["betti0"], color="steelblue")
axes[0].set_xlabel("Cas")
axes[0].set_ylabel("Betti0")
axes[0].set_title("Betti0 par cas")
axes[0].tick_params(axis="x", rotation=45)

# n_gt vs n_pred
x = np.arange(len(df))
axes[1].bar(x - 0.2, df["n_gt"],   width=0.4, label="GT",   color="green",  alpha=0.7)
axes[1].bar(x + 0.2, df["n_pred"], width=0.4, label="Pred", color="tomato", alpha=0.7)
axes[1].set_xticks(x)
axes[1].set_xticklabels(df["case"].str.replace(".nii.gz", ""), rotation=45)
axes[1].set_ylabel("Nombre de composantes connexes")
axes[1].set_title("GT vs Pred — composantes connexes")
axes[1].legend()

plt.tight_layout()
plt.show()

## Visualisation d'un cas — composantes connexes colorées

Chaque couleur = une composante connexe distincte.  
Les composantes présentes dans GT mais absentes de Pred (et vice versa) sont la source directe du Betti0.

In [ ]:
import random

# None = cas avec le Betti0 le plus élevé dans le tableau ci-dessus
# Ou spécifier un nom : CASE_NAME = "PARSE_0091.nii.gz"
CASE_NAME = None

if CASE_NAME is None:
    CASE_NAME = df.iloc[0]["case"]   # pire cas en Betti0

pred_path = os.path.join(PRED_DIR, CASE_NAME)
gt_path   = os.path.join(GT_DIR,   CASE_NAME)

pred_nii = nib.load(pred_path)
gt_nii   = nib.load(gt_path)

pred = pred_nii.get_fdata().astype(bool)
gt   = gt_nii.get_fdata().astype(bool)

labeled_pred, n_pred = label_cc(pred)
labeled_gt,   n_gt   = label_cc(gt)
betti0 = abs(n_pred - n_gt)

print(f"Cas    : {CASE_NAME}")
print(f"Shape  : {gt.shape}")
print(f"GT     : {n_gt} composantes connexes")
print(f"Pred   : {n_pred} composantes connexes")
print(f"Betti0 : {betti0}  ({'pred a trop de composantes' if n_pred > n_gt else 'pred manque des composantes'})")

In [ ]:
def random_colors(n, seed=42):
    """Palette de n couleurs vives distinctes (index 0 = fond noir)."""
    rng = np.random.default_rng(seed)
    cols = rng.uniform(0.35, 1.0, size=(n + 1, 3))
    cols[0] = 0.0
    return cols

def label_to_rgba(labeled_2d, colors, alpha=0.85):
    h, w = labeled_2d.shape
    rgba = np.zeros((h, w, 4))
    for lid in np.unique(labeled_2d):
        if lid == 0:
            continue
        m = labeled_2d == lid
        rgba[m, :3] = colors[lid]
        rgba[m, 3]  = alpha
    return rgba

def component_sizes(labeled_vol):
    sizes = np.bincount(labeled_vol.ravel())
    return sizes[1:]  # exclure le fond

def plot_case(labeled_gt, n_gt, labeled_pred, n_pred, n_slices=6, axis=2):
    depth = labeled_gt.shape[axis]
    slices = np.linspace(int(depth * 0.15), int(depth * 0.85), n_slices, dtype=int)

    colors_gt   = random_colors(n_gt)
    colors_pred = random_colors(n_pred)

    fig, axes = plt.subplots(2, n_slices, figsize=(n_slices * 3, 6.5))

    for col, sl in enumerate(slices):
        sl_gt   = np.take(labeled_gt,   sl, axis=axis)
        sl_pred = np.take(labeled_pred, sl, axis=axis)

        for row, (sl_lab, colors, tag) in enumerate([
            (sl_gt,   colors_gt,   f"GT  z={sl}"),
            (sl_pred, colors_pred, f"Pred  z={sl}"),
        ]):
            rgba = label_to_rgba(sl_lab, colors)
            axes[row, col].imshow(rgba.T, origin="lower", interpolation="nearest")
            axes[row, col].set_title(tag, fontsize=8)
            axes[row, col].axis("off")

    fig.suptitle(
        f"{CASE_NAME}  —  GT: {n_gt} cc  |  Pred: {n_pred} cc  |  Betti0 = {abs(n_pred - n_gt)}",
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()

plot_case(labeled_gt, n_gt, labeled_pred, n_pred)

## Distribution des tailles de composantes

Utile pour comprendre si les composantes "en trop" dans la pred sont de petits artefacts ou de grosses structures mal segmentées.

In [ ]:
sizes_gt   = component_sizes(labeled_gt)
sizes_pred = component_sizes(labeled_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, sizes, label, color in [
    (axes[0], sizes_gt,   f"GT ({n_gt} cc)",   "green"),
    (axes[1], sizes_pred, f"Pred ({n_pred} cc)", "tomato"),
]:
    ax.bar(range(1, len(sizes) + 1), np.sort(sizes)[::-1], color=color, alpha=0.8)
    ax.set_xlabel("Composante (triée par taille desc.)")
    ax.set_ylabel("Taille (voxels)")
    ax.set_title(label)
    ax.set_yscale("log")

fig.suptitle(f"{CASE_NAME} — tailles des composantes connexes", fontsize=12)
plt.tight_layout()
plt.show()

print("GT   — tailles :", np.sort(sizes_gt)[::-1])
print("Pred — tailles :", np.sort(sizes_pred)[::-1])

## Hypothèse : le Betti0 est-il dominé par de petits artefacts ?

Si oui, un simple post-processing (supprimer les composantes < seuil) ferait drastiquement baisser le Betti0 sans impacter les grandes structures.

In [ ]:
def betti0_after_threshold(labeled_vol, n_labels, gt_n, min_size):
    """Betti0 si on supprime les composantes < min_size dans la pred."""
    sizes = np.bincount(labeled_vol.ravel())[1:]
    n_kept = int((sizes >= min_size).sum())
    return abs(n_kept - gt_n), n_kept

thresholds = [1, 10, 50, 100, 200, 500, 1000, 2000]
results = [(t, *betti0_after_threshold(labeled_pred, n_pred, n_gt, t)) for t in thresholds]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot([r[0] for r in results], [r[1] for r in results], "o-", color="steelblue", label="Betti0 après seuil")
ax.axhline(betti0, color="tomato", linestyle="--", label=f"Betti0 original ({betti0})")
ax.set_xlabel("Taille minimale de composante conservée (voxels)")
ax.set_ylabel("Betti0")
ax.set_xscale("log")
ax.set_title(f"{CASE_NAME} — impact d'un seuil de taille sur le Betti0")
ax.legend()
plt.tight_layout()
plt.show()

print(f"{'Seuil':>8}  {'n_pred gardés':>14}  {'Betti0':>8}")
for t, b, n in results:
    print(f"{t:>8}  {n:>14}  {b:>8}")